# Virtual Sites Spanning Residue Boundaries

Test virtual sites that span residue boundaries on proteins, plus ligand virtual sites. Validates PR #430's handling of cross-residue virtual sites.

In [1]:
import copy

import numpy as np
import openmm
import openmm.unit
from openff.toolkit import ForceField as OFFForceField
from openff.toolkit import Molecule, Topology
from openff.toolkit.typing.engines.smirnoff.parameters import VirtualSiteType
from openmm.app import ForceField, Modeller, NoCutoff, PDBFile

from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from openmmforcefields.utils import get_data_filename

In [2]:
# Utility functions (same as notebook 01, with fixed vsite permutation matching)
ENERGY_UNIT = openmm.unit.kilocalories_per_mole
FORCE_UNIT = openmm.unit.kilocalories_per_mole / openmm.unit.angstroms


def compute_energy(system, positions):
    system = copy.deepcopy(system)
    for index, force in enumerate(system.getForces()):
        force.setForceGroup(index)
    platform = openmm.Platform.getPlatformByName("Reference")
    integrator = openmm.VerletIntegrator(0.001)
    context = openmm.Context(system, integrator, platform)
    context.setPositions(positions)
    context.applyConstraints(integrator.getConstraintTolerance())
    energy = {
        "total": context.getState(getEnergy=True).getPotentialEnergy(),
        "components": {
            system.getForce(i).__class__.__name__: context.getState(
                getEnergy=True, groups=(1 << i)
            ).getPotentialEnergy()
            for i in range(system.getNumForces())
        },
    }
    forces = {
        "total": context.getState(getForces=True).getForces(asNumpy=True),
        "components": {
            system.getForce(i).__class__.__name__: context.getState(
                getForces=True, groups=(1 << i)
            ).getForces(asNumpy=True)
            for i in range(system.getNumForces())
        },
    }
    del context, integrator
    return energy, forces


def _get_vsite_parent_particles(system, vsite_idx):
    """Get the parent particle indices for a virtual site."""
    vs = system.getVirtualSite(vsite_idx)
    return tuple(vs.getParticle(j) for j in range(vs.getNumParticles()))


def get_permutation_indices(system_1, system_2):
    """Compute permutation mapping particles from system_1 to system_2.

    Atoms are matched by sequential order. Virtual sites are matched by
    their parent atom identities (translated through the atom mapping),
    NOT by sequential vsite order — the two systems may interleave vsites
    at different positions among the real atoms.
    """
    atoms_1, sites_1, atoms_2, sites_2 = [], [], [], []
    for i in range(system_1.getNumParticles()):
        (sites_1 if system_1.isVirtualSite(i) else atoms_1).append(i)
    for i in range(system_2.getNumParticles()):
        (sites_2 if system_2.isVirtualSite(i) else atoms_2).append(i)

    assert len(atoms_1) == len(atoms_2), f"Atom count mismatch: {len(atoms_1)} vs {len(atoms_2)}"
    assert len(sites_1) == len(sites_2), f"Vsite count mismatch: {len(sites_1)} vs {len(sites_2)}"

    # Atoms map 1:1 by sequential order
    # Build particle_idx -> atom_ordinal maps
    atom_to_ord_1 = {pidx: ord for ord, pidx in enumerate(atoms_1)}
    atom_to_ord_2 = {pidx: ord for ord, pidx in enumerate(atoms_2)}
    ord_to_atom_2 = {ord: pidx for pidx, ord in atom_to_ord_2.items()}

    # Match vsites by their parent atom ordinals
    # For each vsite in sys1, compute its parent atom ordinals, then find
    # the vsite in sys2 with the same parent atom ordinals
    def vsite_parent_ords(system, vsite_idx, atom_to_ord):
        parents = _get_vsite_parent_particles(system, vsite_idx)
        return tuple(atom_to_ord[p] for p in parents)

    sites2_by_parents = {}
    for si in sites_2:
        key = vsite_parent_ords(system_2, si, atom_to_ord_2)
        sites2_by_parents.setdefault(key, []).append(si)

    # Build matched pairs
    site_pairs = []  # (sys1_idx, sys2_idx)
    for si in sites_1:
        key = vsite_parent_ords(system_1, si, atom_to_ord_1)
        matched = sites2_by_parents.get(key, [])
        assert matched, f"No matching vsite in system_2 for vsite {si} with parent atoms {key}"
        site_pairs.append((si, matched.pop(0)))

    # Build full permutation
    n = system_1.getNumParticles()
    perm_12 = [-1] * n
    perm_21 = [-1] * n
    for a1, a2 in zip(atoms_1, atoms_2):
        perm_12[a1] = a2
        perm_21[a2] = a1
    for s1, s2 in site_pairs:
        perm_12[s1] = s2
        perm_21[s2] = s1

    return perm_12, perm_21


def compare_energies(name, positions, template_system, reference_system):
    forces_perm, pos_perm = get_permutation_indices(template_system, reference_system)
    template_energy, template_forces = compute_energy(template_system, positions)
    ref_positions = [positions[i] for i in pos_perm]
    ref_energy, ref_forces = compute_energy(reference_system, ref_positions)
    ref_forces["total"] = (
        np.array([ref_forces["total"][i].value_in_unit(FORCE_UNIT) for i in forces_perm]) * FORCE_UNIT
    )
    for comp, f in ref_forces["components"].items():
        ref_forces["components"][comp] = (
            np.array([f[i].value_in_unit(FORCE_UNIT) for i in forces_perm]) * FORCE_UNIT
        )
    components = set(ref_energy["components"]) & set(template_energy["components"])
    print(f"\n{'Component':<28} {'Template':>20} {'Reference':>20} {'Delta':>14}")
    print("-" * 84)
    for key in sorted(components):
        t_e = template_energy["components"][key].value_in_unit(ENERGY_UNIT)
        r_e = ref_energy["components"][key].value_in_unit(ENERGY_UNIT)
        print(f"{key:<28} {t_e:20.6f} {r_e:20.6f} {t_e - r_e:14.6e}")
    t_tot = template_energy["total"].value_in_unit(ENERGY_UNIT)
    r_tot = ref_energy["total"].value_in_unit(ENERGY_UNIT)
    delta = template_energy["total"] - ref_energy["total"]
    print("-" * 84)
    print(f"{'TOTAL':<28} {t_tot:20.6f} {r_tot:20.6f} {(t_tot - r_tot):14.6e}")

    def norm_sq(x):
        return (x * x).sum() / x.shape[0]
    def relative_deviation(x, y):
        x = x.value_in_unit(FORCE_UNIT)
        y = y.value_in_unit(FORCE_UNIT)
        scale = norm_sq(x) + norm_sq(y)
        return np.sqrt(norm_sq(x - y) / scale) if scale > 0 else 0

    rel_dev = relative_deviation(template_forces["total"], ref_forces["total"])
    print(f"\nRelative force deviation: {rel_dev:.2e}")
    assert abs(delta) <= 1.0e-8 * ENERGY_UNIT, f"Energy deviation {delta} too large"
    assert rel_dev <= 1.0e-8, f"Force deviation {rel_dev} too large"
    print(f"PASS: {name}")


def propagate_dynamics(positions, system, nsteps=100):
    integrator = openmm.LangevinIntegrator(
        300 * openmm.unit.kelvin, 1.0 / openmm.unit.picoseconds, 1.0 * openmm.unit.femtoseconds
    )
    platform = openmm.Platform.getPlatformByName("Reference")
    context = openmm.Context(system, integrator, platform)
    context.setPositions(positions)
    openmm.LocalEnergyMinimizer.minimize(context)
    integrator.step(nsteps)
    return context.getState(getPositions=True).getPositions()


def count_vsites(system):
    return sum(system.isVirtualSite(i) for i in range(system.getNumParticles()))

## Build Force Field with Custom Virtual Site Parameters

In [3]:
from openff.units import Quantity as OFFQuantity
def make_ff_with_vsites():
    """Create an OpenFF force field with custom virtual site parameters."""
    openff_ff = OFFForceField("openff_no_water-3.0.0-alpha0.offxml")
    openff_ff.get_parameter_handler("VirtualSites")

    # TrivalentLonePair on backbone N: N bonded to Calpha (same residue), H,
    # and C=O (adjacent residue) -- this is the cross-residue case
    openff_ff["VirtualSites"].add_parameter(
        parameter=VirtualSiteType(
            smirks="[#6X4:2][#7X3:1]([#1:3])[#6X3:4]",
            type="TrivalentLonePair",
            match="once",
            distance="0.5 * angstrom ** 1",
            #outOfPlaneAngle=OFFQuantity(90.0, "degree"),
            #outOfPlaneAngle="None",
            inPlaneAngle="None",
            epsilon="1.0 * kilocalories_per_mole",
            sigma="1.5 * angstrom",
            charge_increment1="0.05 * elementary_charge ** 1",
            charge_increment2="-0.08 * elementary_charge ** 1",
            charge_increment3="-0.06 * elementary_charge ** 1",
            charge_increment4="-0.03 * elementary_charge ** 1",
        ),
    )

    # MonovalentLonePair on carbonyl O: O=C-N where N is in adjacent residue
    openff_ff["VirtualSites"].add_parameter(
        parameter=VirtualSiteType(
            smirks="[#8:1]=[#6X3+0:2]-[#7:3]",
            type="MonovalentLonePair",
            #match="once",
            match="all_permutations",
            distance="0.6 * angstrom ** 1",
            #outOfPlaneAngle="0.1 * degree ** 1",
            outOfPlaneAngle=OFFQuantity(0.0, "degree"),
            inPlaneAngle="120.0 * degree ** 1",
            epsilon="0.5 * kilocalories_per_mole",
            sigma="1.6 * angstrom",
            charge_increment1="0.05 * elementary_charge ** 1",
            charge_increment2="-0.02 * elementary_charge ** 1",
            charge_increment3="-0.04 * elementary_charge ** 1",
        ),
    )

    # BondCharge on Cl (for ligand testing)
    openff_ff["VirtualSites"].add_parameter(
        parameter=VirtualSiteType(
            smirks="[#6:2]-[#17X1:1]",
            type="BondCharge",
            #match="once",
            match="all_permutations",
            distance="0.4 * angstrom ** 1",
            epsilon="0.3 * kilocalories_per_mole",
            sigma="1.7 * angstrom",
            charge_increment1="0.03 * elementary_charge ** 1",
            charge_increment2="-0.05 * elementary_charge ** 1",
        ),
    )

    return openff_ff


openff_ff = make_ff_with_vsites()
print(f"Force field has {len(openff_ff['VirtualSites'].parameters)} virtual site parameters")

Force field has 3 virtual site parameters


## Test 1: Protein-Only with test-ala-3.pdb

In [4]:
data_file = get_data_filename("test-ala-3.pdb")
pdb = PDBFile(data_file)
off_top = Topology.from_pdb(data_file)

# Interchange (reference) path
interchange_system = openff_ff.create_openmm_system(off_top)
n_vsites_interchange = count_vsites(interchange_system)
print(f"Interchange system: {interchange_system.getNumParticles()} particles, {n_vsites_interchange} virtual sites")

# OMMFFS (template generator) path with reversed atom order
molecule = off_top.molecule(0)
molecule = molecule.remap({i: molecule.n_atoms - i - 1 for i in range(molecule.n_atoms)})

generator = SMIRNOFFTemplateGenerator(molecules=[molecule], forcefield=openff_ff.to_string())
openmm_ff = ForceField()
openmm_ff.registerTemplateGenerator(generator.generator)

modeller = Modeller(pdb.topology, pdb.positions)
modeller.addExtraParticles(openmm_ff)
ommffs_system = openmm_ff.createSystem(modeller.topology, nonbondedMethod=NoCutoff)
n_vsites_ommffs = count_vsites(ommffs_system)
print(f"OMMFFS system:      {ommffs_system.getNumParticles()} particles, {n_vsites_ommffs} virtual sites")

assert n_vsites_interchange > 0, "Expected virtual sites in Interchange system"
assert n_vsites_interchange == n_vsites_ommffs, (
    f"Vsite count mismatch: Interchange={n_vsites_interchange}, OMMFFS={n_vsites_ommffs}"
)
print(f"\nVirtual site counts match: {n_vsites_interchange}")

/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


[14:12:34] WARNING: Proton(s) added/removed



Interchange system: 37 particles, 4 virtual sites


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


OMMFFS system:      37 particles, 4 virtual sites

Virtual site counts match: 4


[14:12:35] WARNING: Proton(s) added/removed



In [5]:
# Debug: diff the two OpenMM systems, matching vsites by parent atoms
import xml.etree.ElementTree as ET
from io import StringIO


def _build_atom_maps(system):
    """
    Returns:
      particle_to_atom: dict mapping particle index -> atom ordinal (None for vsites)
      atom_to_particle: dict mapping atom ordinal -> particle index
      vsite_label: dict mapping vsite particle index -> "vsN" label (ordered by appearance)
    """
    particle_to_atom = {}
    atom_to_particle = {}
    vsite_label = {}
    atom_ord = 0
    vsite_ord = 0
    for i in range(system.getNumParticles()):
        if system.isVirtualSite(i):
            particle_to_atom[i] = None
            vsite_label[i] = f"vs{vsite_ord}"
            vsite_ord += 1
        else:
            particle_to_atom[i] = atom_ord
            atom_to_particle[atom_ord] = i
            atom_ord += 1
    return particle_to_atom, atom_to_particle, vsite_label


def _vsite_parent_atoms(system, vsite_idx, particle_to_atom):
    """Get the parent atom ordinals (not particle indices) for a vsite."""
    vs = system.getVirtualSite(vsite_idx)
    parent_particles = [vs.getParticle(j) for j in range(vs.getNumParticles())]
    parent_atoms = tuple(particle_to_atom[p] for p in parent_particles)
    return parent_atoms


def _vsite_details(system, vsite_idx):
    """Extract type and geometry details for a vsite."""
    vs = system.getVirtualSite(vsite_idx)
    info = {"type": vs.__class__.__name__}
    if isinstance(vs, openmm.LocalCoordinatesSite):
        info["originWeights"] = list(vs.getOriginWeights())
        info["xWeights"] = list(vs.getXWeights())
        info["yWeights"] = list(vs.getYWeights())
        # getLocalPosition() returns a Vec3 (Quantity in nm) — strip units
        lp = vs.getLocalPosition()
        info["localPosition"] = [lp.x, lp.y, lp.z]
    return info


def _nb_params(system, particle_idx):
    """Get nonbonded params for a particle."""
    nb = [f for f in system.getForces() if isinstance(f, openmm.NonbondedForce)][0]
    q, sig, eps = nb.getParticleParameters(particle_idx)
    return {
        "charge": q.value_in_unit(openmm.unit.elementary_charge),
        "sigma": sig.value_in_unit(openmm.unit.angstrom),
        "epsilon": eps.value_in_unit(openmm.unit.kilocalories_per_mole),
    }


def compare_systems(sys1, sys2, name1="OMMFFS", name2="Interchange"):
    p2a_1, a2p_1, vslabel_1 = _build_atom_maps(sys1)
    p2a_2, a2p_2, vslabel_2 = _build_atom_maps(sys2)

    n_atoms_1 = len(a2p_1)
    n_atoms_2 = len(a2p_2)
    vsites_1 = [i for i in range(sys1.getNumParticles()) if sys1.isVirtualSite(i)]
    vsites_2 = [i for i in range(sys2.getNumParticles()) if sys2.isVirtualSite(i)]

    print(f"Atoms: {name1}={n_atoms_1}, {name2}={n_atoms_2}")
    print(f"Vsites: {name1}={len(vsites_1)}, {name2}={len(vsites_2)}")

    # Build lookup: parent_atoms_tuple -> vsite particle index for each system
    vs_by_parents_1 = {}
    for vi in vsites_1:
        key = _vsite_parent_atoms(sys1, vi, p2a_1)
        vs_by_parents_1.setdefault(key, []).append(vi)

    vs_by_parents_2 = {}
    for vi in vsites_2:
        key = _vsite_parent_atoms(sys2, vi, p2a_2)
        vs_by_parents_2.setdefault(key, []).append(vi)

    all_parent_keys = sorted(set(vs_by_parents_1.keys()) | set(vs_by_parents_2.keys()))

    print(f"\n{'Parent atoms':<30} {name1 + ' idx':>12} {name2 + ' idx':>12} {'Type match':>12}")
    print("=" * 80)

    # Build a mapping from vsite particle index -> matched partner for exception comparison
    vsite_match_1to2 = {}  # sys1 vsite particle idx -> sys2 vsite particle idx

    for parent_key in all_parent_keys:
        list1 = vs_by_parents_1.get(parent_key, [])
        list2 = vs_by_parents_2.get(parent_key, [])

        max_len = max(len(list1), len(list2))
        for k in range(max_len):
            idx1 = list1[k] if k < len(list1) else None
            idx2 = list2[k] if k < len(list2) else None

            if idx1 is not None and idx2 is not None:
                vsite_match_1to2[idx1] = idx2

            det1 = _vsite_details(sys1, idx1) if idx1 is not None else None
            det2 = _vsite_details(sys2, idx2) if idx2 is not None else None

            type1 = det1["type"] if det1 else "MISSING"
            type2 = det2["type"] if det2 else "MISSING"
            type_match = "OK" if type1 == type2 else "MISMATCH"

            idx1_str = str(idx1) if idx1 is not None else "---"
            idx2_str = str(idx2) if idx2 is not None else "---"
            print(f"{str(parent_key):<30} {idx1_str:>12} {idx2_str:>12} {type_match:>12}")

            # Compare geometry
            if det1 and det2 and type1 == type2:
                for field in ["originWeights", "xWeights", "yWeights", "localPosition"]:
                    if field in det1:
                        v1 = det1[field]
                        v2 = det2[field]
                        match = all(abs(float(a) - float(b)) < 1e-10 for a, b in zip(v1, v2))
                        if not match:
                            print(f"    {field}: *** DIFF ***")
                            print(f"      {name1}: {[f'{float(x):.8f}' for x in v1]}")
                            print(f"      {name2}: {[f'{float(x):.8f}' for x in v2]}")

            # Compare nonbonded params
            if idx1 is not None and idx2 is not None:
                nb1 = _nb_params(sys1, idx1)
                nb2 = _nb_params(sys2, idx2)
                diffs = []
                for param in ["charge", "sigma", "epsilon"]:
                    if abs(nb1[param] - nb2[param]) > 1e-6:
                        diffs.append(f"{param}: {name1}={nb1[param]:.6f} {name2}={nb2[param]:.6f}")
                if diffs:
                    print(f"    Nonbonded diffs: {'; '.join(diffs)}")

    # Also check for nonbonded diffs on real atoms
    print(f"\n--- Real atom nonbonded diffs ---")
    atom_diffs = 0
    for atom_ord in range(n_atoms_1):
        pi1 = a2p_1[atom_ord]
        pi2 = a2p_2[atom_ord]
        nb1 = _nb_params(sys1, pi1)
        nb2 = _nb_params(sys2, pi2)
        diffs = []
        for param in ["charge", "sigma", "epsilon"]:
            if abs(nb1[param] - nb2[param]) > 1e-6:
                diffs.append(f"{param}: {name1}={nb1[param]:.6f} {name2}={nb2[param]:.6f}")
        if diffs:
            atom_diffs += 1
            print(f"  Atom {atom_ord} ({name1} particle {pi1}, {name2} particle {pi2}): {'; '.join(diffs)}")
    if atom_diffs == 0:
        print("  None")

    # Compare nonbonded exceptions involving vsites
    print(f"\n--- Nonbonded exception diffs (vsite-involved) ---")
    nb_force_1 = [f for f in sys1.getForces() if isinstance(f, openmm.NonbondedForce)][0]
    nb_force_2 = [f for f in sys2.getForces() if isinstance(f, openmm.NonbondedForce)][0]

    # Build a unified particle mapping: sys1 particle idx -> canonical label
    # For atoms: the atom ordinal (int). For vsites: the matched pair label.
    def canonical_label_1(pidx):
        if p2a_1[pidx] is not None:
            return ("atom", p2a_1[pidx])
        return ("vsite", _vsite_parent_atoms(sys1, pidx, p2a_1))

    def canonical_label_2(pidx):
        if p2a_2[pidx] is not None:
            return ("atom", p2a_2[pidx])
        return ("vsite", _vsite_parent_atoms(sys2, pidx, p2a_2))

    def get_exceptions_canonical(nb_force, system, label_fn):
        excs = {}
        vsite_set = {i for i in range(system.getNumParticles()) if system.isVirtualSite(i)}
        for ei in range(nb_force.getNumExceptions()):
            p1, p2, chg, sig, eps = nb_force.getExceptionParameters(ei)
            if p1 in vsite_set or p2 in vsite_set:
                l1 = label_fn(p1)
                l2 = label_fn(p2)
                key = (l1, l2) if l1 <= l2 else (l2, l1)
                excs[key] = (
                    chg.value_in_unit(openmm.unit.elementary_charge**2),
                    sig.value_in_unit(openmm.unit.angstrom),
                    eps.value_in_unit(openmm.unit.kilocalories_per_mole),
                )
        return excs

    exc1 = get_exceptions_canonical(nb_force_1, sys1, canonical_label_1)
    exc2 = get_exceptions_canonical(nb_force_2, sys2, canonical_label_2)

    all_exc_keys = sorted(set(exc1.keys()) | set(exc2.keys()), key=str)
    exc_diffs = 0
    for key in all_exc_keys:
        e1 = exc1.get(key)
        e2 = exc2.get(key)
        if e1 is None:
            print(f"  {key}: only in {name2}: chgProd={e2[0]:.6f} sig={e2[1]:.4f} eps={e2[2]:.6f}")
            exc_diffs += 1
        elif e2 is None:
            print(f"  {key}: only in {name1}: chgProd={e1[0]:.6f} sig={e1[1]:.4f} eps={e1[2]:.6f}")
            exc_diffs += 1
        else:
            diffs = []
            labels = ["chgProd", "sigma", "epsilon"]
            for i, lbl in enumerate(labels):
                if abs(e1[i] - e2[i]) > 1e-6:
                    diffs.append(f"{lbl}: {name1}={e1[i]:.6f} {name2}={e2[i]:.6f}")
            if diffs:
                print(f"  {key}: {'; '.join(diffs)}")
                exc_diffs += 1
    if exc_diffs == 0:
        print("  None")


compare_systems(ommffs_system, interchange_system)

Atoms: OMMFFS=33, Interchange=33
Vsites: OMMFFS=4, Interchange=4

Parent atoms                     OMMFFS idx Interchange idx   Type match
(11, 10, 12)                             12           35           OK
(12, 14, 13, 10)                         23           33           OK
(21, 20, 25)                             24           36           OK
(25, 27, 26, 20)                         36           34           OK

--- Real atom nonbonded diffs ---
  None

--- Nonbonded exception diffs (vsite-involved) ---
  None


## Test 2: Protein + Ligand (chloroacetamide)

In [6]:
# Create ligand molecule: chloroacetamide
ligand = Molecule.from_smiles("ClCC(=O)N")
ligand.generate_conformers(n_conformers=1)
print(f"Ligand: {ligand.to_smiles()} ({ligand.n_atoms} atoms)")

# Build combined topology
combined_top = Topology.from_pdb(data_file)
combined_top.add_molecule(ligand)

# Interchange reference
interchange_system_combined = openff_ff.create_openmm_system(combined_top)
n_vsites_ref = count_vsites(interchange_system_combined)
print(f"\nInterchange combined: {interchange_system_combined.getNumParticles()} particles, {n_vsites_ref} vsites")

# OMMFFS path: register both protein and ligand molecules
protein_mol = off_top.molecule(0)
generator2 = SMIRNOFFTemplateGenerator(
    molecules=[protein_mol, ligand],
    forcefield=openff_ff.to_string(),
)
openmm_ff2 = ForceField()
openmm_ff2.registerTemplateGenerator(generator2.generator)

# Build OpenMM topology with ligand appended
modeller2 = Modeller(pdb.topology, pdb.positions)
# Add ligand to the OpenMM topology
ligand_top = ligand.to_topology().to_openmm()
ligand_positions = ligand.conformers[0].to_openmm()
modeller2.add(ligand_top, ligand_positions)
modeller2.addExtraParticles(openmm_ff2)

ommffs_system_combined = openmm_ff2.createSystem(modeller2.topology, nonbondedMethod=NoCutoff)
n_vsites_ommffs2 = count_vsites(ommffs_system_combined)
print(f"OMMFFS combined:     {ommffs_system_combined.getNumParticles()} particles, {n_vsites_ommffs2} vsites")

assert n_vsites_ref > 0
assert n_vsites_ref == n_vsites_ommffs2, (
    f"Vsite count mismatch: {n_vsites_ref} vs {n_vsites_ommffs2}"
)
print(f"Virtual site counts match: {n_vsites_ref}")

Ligand: [H][N]([H])[C](=[O])[C]([H])([H])[Cl] (9 atoms)


[14:12:36] WARNING: Proton(s) added/removed




Interchange combined: 48 particles, 6 vsites


[14:12:37] WARNING: Proton(s) added/removed



OMMFFS combined:     48 particles, 6 vsites
Virtual site counts match: 6


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openmm/vec3.py:77: RuntimeWarning: invalid value encountered in scalar divide
  return Vec3(self.x/other, self.y/other, self.z/other)


In [7]:
# Minimal reproducer: OMMFFS adds a spurious constraint when a template is loaded
# mid-createSystem via the generator callback, and additional residues follow.
#
# The bug is NOT specific to multi-residue molecules. It triggers whenever:
#   1. The first molecule has >1 heavy atom (so it has a heavy-heavy bond)
#   2. A different second molecule follows in the topology
#   3. Templates are loaded via the generator callback during createSystem
#
# The first heavy-heavy bond in the HarmonicBondForce section of the first
# molecule's FFXML template is erroneously promoted to a constraint.

from openmm import Vec3
import io

ff_nv = OFFForceField("openff_no_water-3.0.0-alpha0.offxml")

# === Simplest reproducer: ethanol + methane (no peptide needed) ===
mol_ethanol = Molecule.from_smiles("CCO")
mol_ethanol.generate_conformers(n_conformers=1)
mol_methane = Molecule.from_smiles("C")
mol_methane.generate_conformers(n_conformers=1)

def make_two_mol_modeller(mol_a, mol_b):
    top_a = mol_a.to_topology().to_openmm()
    pos_a = mol_a.conformers[0].to_openmm()
    mod = Modeller(top_a, pos_a)
    raw_b = mol_b.conformers[0].to_openmm().value_in_unit(openmm.unit.nanometer)
    pos_b = [Vec3(x + 2.0, y, z) for x, y, z in raw_b] * openmm.unit.nanometer
    mod.add(mol_b.to_topology().to_openmm(), pos_b)
    return mod

def count_extra_constraints(ic_sys, of_sys):
    ic_pairs = set()
    for ci in range(ic_sys.getNumConstraints()):
        p1, p2, _ = ic_sys.getConstraintParameters(ci)
        ic_pairs.add((min(p1, p2), max(p1, p2)))
    of_pairs = set()
    for ci in range(of_sys.getNumConstraints()):
        p1, p2, _ = of_sys.getConstraintParameters(ci)
        of_pairs.add((min(p1, p2), max(p1, p2)))
    extra = of_pairs - ic_pairs
    details = []
    for p1, p2 in sorted(extra):
        m1 = of_sys.getParticleMass(p1).value_in_unit(openmm.unit.dalton)
        m2 = of_sys.getParticleMass(p2).value_in_unit(openmm.unit.dalton)
        details.append(f"p{p1}({m1:.0f}Da)-p{p2}({m2:.0f}Da)")
    return len(extra), details

print("=== Constraint Bug Reproducer ===\n")
print(f"{'Case':<50} {'IC':>4} {'OF':>4} {'Extra':>6}  Details")
print("-" * 85)

cases = [
    ("ethanol + methane", "CCO", "C"),
    ("methane + ethanol", "C", "CCO"),
    ("ethane + methane", "CC", "C"),
    ("methane + ethane", "C", "CC"),
    ("methane + methane", "C", "C"),
    ("ethanol alone", "CCO", None),
    ("methane alone", "C", None),
]

for label, smi1, smi2 in cases:
    m1 = Molecule.from_smiles(smi1)
    m1.generate_conformers(n_conformers=1)
    mols = [m1]
    ct = Topology()
    ct.add_molecule(m1)

    m2 = None
    if smi2:
        m2 = Molecule.from_smiles(smi2)
        m2.generate_conformers(n_conformers=1)
        mols.append(m2)
        ct.add_molecule(m2)

    ic_sys = ff_nv.create_openmm_system(ct)

    gen = SMIRNOFFTemplateGenerator(molecules=mols, forcefield="openff_no_water-3.0.0-alpha0.offxml")
    omm_ff = ForceField()
    omm_ff.registerTemplateGenerator(gen.generator)
    top1 = m1.to_topology().to_openmm()
    pos1 = m1.conformers[0].to_openmm()
    mod = Modeller(top1, pos1)
    if m2:
        raw2 = m2.conformers[0].to_openmm().value_in_unit(openmm.unit.nanometer)
        pos2 = [Vec3(x + 2.0, y, z) for x, y, z in raw2] * openmm.unit.nanometer
        mod.add(m2.to_topology().to_openmm(), pos2)
    of_sys = omm_ff.createSystem(mod.topology, nonbondedMethod=NoCutoff)

    ic_c = ic_sys.getNumConstraints()
    of_c = of_sys.getNumConstraints()
    n_extra, extra_details = count_extra_constraints(ic_sys, of_sys)
    marker = "  <-- BUG" if n_extra > 0 else ""
    detail_str = ", ".join(extra_details) if extra_details else ""
    print(f"{label:50} {ic_c:4d} {of_c:4d} {of_c - ic_c:+5d}{marker}  {detail_str}")

print()
print("The bug triggers when:")
print("  1. The first molecule has >1 heavy atom (has a heavy-heavy bond)")
print("  2. A second, different molecule follows in the topology")
print("  3. Templates are loaded via the generator callback during createSystem")
print()
print("The first heavy-heavy bond in the FFXML is erroneously promoted to a constraint.")
print("This is an OpenMM ForceField bug, not specific to openmmforcefields.")

=== Constraint Bug Reproducer ===

Case                                                 IC   OF  Extra  Details
-------------------------------------------------------------------------------------


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


ethanol + methane                                    10   10    +0  


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


...

/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


ethanol alone                                         6    6    +0  


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


methane alone                                         4    4    +0  

The bug triggers when:
  1. The first molecule has >1 heavy atom (has a heavy-heavy bond)
  2. A second, different molecule follows in the topology
  3. Templates are loaded via the generator callback during createSystem

The first heavy-heavy bond in the FFXML is erroneously promoted to a constraint.
This is an OpenMM ForceField bug, not specific to openmmforcefields.


In [8]:
# Energy comparison for the combined system.
# Use the pre-load workaround to avoid the constraint bug documented in cell-10,
# so we can test that the actual force field parameters (bonds, angles, torsions,
# nonbonded, vsites) are identical between the two pathways.
#
# Use the test suite's standard tolerances (1e-2 kcal/mol energy, 5e-5 relative
# force) since the combined system with vsites on both protein and ligand has
# slightly larger numerical deviations than the protein-only case.

import io as _io

ENERGY_TOL_COMBINED = 1.0e-2 * ENERGY_UNIT
FORCE_TOL_COMBINED = 5.0e-5

protein_mol_for_fix = off_top.molecule(0)
gen_combined_fix = SMIRNOFFTemplateGenerator(
    molecules=[protein_mol_for_fix, ligand],
    forcefield=openff_ff.to_string(),
)
omm_ff_combined_fix = ForceField()
omm_ff_combined_fix.registerTemplateGenerator(gen_combined_fix.generator)
# Pre-load templates to avoid the mid-createSystem constraint bug
pep_xml = gen_combined_fix.generate_residue_template(protein_mol_for_fix)
lig_xml = gen_combined_fix.generate_residue_template(ligand)
omm_ff_combined_fix.loadFile(_io.StringIO(pep_xml))
omm_ff_combined_fix.loadFile(_io.StringIO(lig_xml))

modeller_fix = Modeller(pdb.topology, pdb.positions)
ligand_top_fix = ligand.to_topology().to_openmm()
ligand_positions_fix = ligand.conformers[0].to_openmm()
modeller_fix.add(ligand_top_fix, ligand_positions_fix)
modeller_fix.addExtraParticles(omm_ff_combined_fix)
ommffs_fixed = omm_ff_combined_fix.createSystem(modeller_fix.topology, nonbondedMethod=NoCutoff)

print(f"OMMFFS (pre-loaded): {ommffs_fixed.getNumParticles()} particles, "
      f"{count_vsites(ommffs_fixed)} vsites, "
      f"{ommffs_fixed.getNumConstraints()} constraints")
print(f"Interchange:         {interchange_system_combined.getNumParticles()} particles, "
      f"{count_vsites(interchange_system_combined)} vsites, "
      f"{interchange_system_combined.getNumConstraints()} constraints")

# Use a custom compare with relaxed tolerances for the combined system
forces_perm, pos_perm = get_permutation_indices(ommffs_fixed, interchange_system_combined)
new_pos = propagate_dynamics(modeller_fix.positions, ommffs_fixed)
template_energy, template_forces = compute_energy(ommffs_fixed, new_pos)
ref_positions = [new_pos[i] for i in pos_perm]
ref_energy, ref_forces = compute_energy(interchange_system_combined, ref_positions)
ref_forces["total"] = (
    np.array([ref_forces["total"][i].value_in_unit(FORCE_UNIT) for i in forces_perm]) * FORCE_UNIT
)

components = set(ref_energy["components"]) & set(template_energy["components"])
print(f"\n{'Component':<28} {'OMMFFS':>20} {'Interchange':>20} {'Delta':>14}")
print("-" * 84)
for key in sorted(components):
    t_e = template_energy["components"][key].value_in_unit(ENERGY_UNIT)
    r_e = ref_energy["components"][key].value_in_unit(ENERGY_UNIT)
    print(f"{key:<28} {t_e:20.6f} {r_e:20.6f} {t_e - r_e:14.6e}")
t_tot = template_energy["total"].value_in_unit(ENERGY_UNIT)
r_tot = ref_energy["total"].value_in_unit(ENERGY_UNIT)
delta = template_energy["total"] - ref_energy["total"]
print("-" * 84)
print(f"{'TOTAL':<28} {t_tot:20.6f} {r_tot:20.6f} {(t_tot - r_tot):14.6e}")

def norm_sq(x):
    return (x * x).sum() / x.shape[0]
def relative_deviation(x, y):
    x = x.value_in_unit(FORCE_UNIT)
    y = y.value_in_unit(FORCE_UNIT)
    scale = norm_sq(x) + norm_sq(y)
    return np.sqrt(norm_sq(x - y) / scale) if scale > 0 else 0

rel_dev = relative_deviation(template_forces["total"], ref_forces["total"])
print(f"\nRelative force deviation: {rel_dev:.2e}")
assert abs(delta) <= ENERGY_TOL_COMBINED, f"Energy deviation {delta} exceeds {ENERGY_TOL_COMBINED}"
assert rel_dev <= FORCE_TOL_COMBINED, f"Force deviation {rel_dev} exceeds {FORCE_TOL_COMBINED}"
print("PASS: ala-3 + ClCC(=O)N vsites (pre-loaded, standard tolerances)")

/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


[14:12:58] WARNING: Proton(s) added/removed



OMMFFS (pre-loaded): 48 particles, 6 vsites, 21 constraints
Interchange:         48 particles, 6 vsites, 21 constraints

Component                                  OMMFFS          Interchange          Delta
------------------------------------------------------------------------------------
CMMotionRemover                          0.000000             0.000000   0.000000e+00
HarmonicAngleForce                       5.501574             5.501574   0.000000e+00
HarmonicBondForce                        1.894050             1.894050  -2.220446e-16
NonbondedForce                         -38.240355           -38.240355  -6.394885e-14
PeriodicTorsionForce                     4.163795             4.163795   0.000000e+00
------------------------------------------------------------------------------------
TOTAL                                  -26.680935           -26.680935  -6.750156e-14

Relative force deviation: 4.72e-16
PASS: ala-3 + ClCC(=O)N vsites (pre-loaded, standard tolerances)


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openmm/vec3.py:77: RuntimeWarning: invalid value encountered in scalar divide
  return Vec3(self.x/other, self.y/other, self.z/other)


## Inspect Virtual Sites: Identify Cross-Residue Spanning

In [9]:
# Inspect the OMMFFS system to identify vsites and their parent atoms
print("Virtual sites in OMMFFS protein system (test-ala-3):")
print(f"{'Particle':>10} {'Type':>20} {'Parent atoms':>30}")
print("-" * 62)

for i in range(ommffs_system.getNumParticles()):
    if ommffs_system.isVirtualSite(i):
        vsite = ommffs_system.getVirtualSite(i)
        vsite_type = vsite.__class__.__name__
        parent_indices = [vsite.getParticle(j) for j in range(vsite.getNumParticles())]
        print(f"{i:10d} {vsite_type:>20} {str(parent_indices):>30}")

print(f"\nTotal virtual sites: {count_vsites(ommffs_system)}")

Virtual sites in OMMFFS protein system (test-ala-3):
  Particle                 Type                   Parent atoms
--------------------------------------------------------------
        12 LocalCoordinatesSite                   [11, 10, 13]
        23 LocalCoordinatesSite               [13, 15, 14, 10]
        24 LocalCoordinatesSite                   [22, 21, 28]
        36 LocalCoordinatesSite               [28, 30, 29, 21]

Total virtual sites: 4


In [10]:
print("All virtual site cross-residue tests passed!")

All virtual site cross-residue tests passed!


In [11]:
import io
from openmm.app import ForceField, Topology, NoCutoff, Element

MOL_A = """<ForceField>
 <AtomTypes>
  <Type name="A0" mass="12" class="A0" element="C"/>
  <Type name="A1" mass="12" class="A1" element="C"/>
 </AtomTypes>
 <HarmonicBondForce>
  <Bond class1="A0" class2="A1" length="0.15" k="200000"/>
 </HarmonicBondForce>
 <NonbondedForce coulomb14scale="1" lj14scale="1">
  <Atom class="A0" sigma="0.3" epsilon="0.1" charge="0"/>
  <Atom class="A1" sigma="0.3" epsilon="0.1" charge="0"/>
 </NonbondedForce>
 <Residues>
  <Residue name="MOLA">
   <Atom name="a0" type="A0"/><Atom name="a1" type="A1"/>
   <Bond atomName1="a0" atomName2="a1"/>
  </Residue>
 </Residues>
</ForceField>"""

MOL_B = """<ForceField>
 <AtomTypes>
  <Type name="B0" mass="12" class="B0" element="C"/>
  <Type name="B1" mass="1" class="B1" element="H"/>
 </AtomTypes>
 <HarmonicBondForce>
  <Bond class1="B0" class2="B1" length="0.11" k="300000"/>
 </HarmonicBondForce>
 <NonbondedForce coulomb14scale="1" lj14scale="1">
  <Atom class="B0" sigma="0.3" epsilon="0.1" charge="0"/>
  <Atom class="B1" sigma="0.1" epsilon="0.01" charge="0"/>
 </NonbondedForce>
 <Residues>
  <Residue name="MOLB">
   <Atom name="b0" type="B0"/><Atom name="b1" type="B1"/>
   <Bond atomName1="b0" atomName2="b1"/>
   <Constraint atomName1="b0" atomName2="b1" distance="0.11"/>
  </Residue>
 </Residues>
</ForceField>"""

top = Topology()
c = top.addChain()
C, H = Element.getBySymbol('C'), Element.getBySymbol('H')
r1 = top.addResidue('MOLA', c)
a0, a1 = top.addAtom('a0', C, r1), top.addAtom('a1', C, r1)
top.addBond(a0, a1)
r2 = top.addResidue('MOLB', c)
b0, b1 = top.addAtom('b0', C, r2), top.addAtom('b1', H, r2)
top.addBond(b0, b1)

ff = ForceField()
ff.loadFile(io.StringIO(MOL_A))
ff.loadFile(io.StringIO(MOL_B))
sys = ff.createSystem(top, 
                      nonbondedMethod=NoCutoff,
                      constraints=None)

print(f"Constraints: {sys.getNumConstraints()}")  # Expected 1, got 2
for i in range(sys.getNumConstraints()):
    p1, p2, d = sys.getConstraintParameters(i)
    print(f"  ({p1}, {p2})  d={d}")
# (0, 1)  d=0.15 nm   <-- SPURIOUS: MOLA's C-C bond, not a real constraint
# (2, 3)  d=0.11 nm   <-- correct: MOLB's C-H constraint


Constraints: 1
  (2, 3)  d=0.11 nm
